## PS-02: Real-Time Algorithmic Fraud & Anomaly Detection in Streaming Data

In [1]:
import pandas as pd
import numpy as np


In [2]:
file_path = "../dataset/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path, nrows=10000)

In [3]:
df.shape

(10000, 11)

In [4]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   step            10000 non-null  int64  
 1   type            10000 non-null  str    
 2   amount          10000 non-null  float64
 3   nameOrig        10000 non-null  str    
 4   oldbalanceOrg   10000 non-null  float64
 5   newbalanceOrig  10000 non-null  float64
 6   nameDest        10000 non-null  str    
 7   oldbalanceDest  10000 non-null  float64
 8   newbalanceDest  10000 non-null  float64
 9   isFraud         10000 non-null  int64  
 10  isFlaggedFraud  10000 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 859.5 KB


In [7]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["isFraud"].value_counts()

isFraud
0    9932
1      68
Name: count, dtype: int64

In [10]:
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.32
1     0.68
Name: proportion, dtype: float64

In [11]:
df["type"].value_counts()

type
PAYMENT     5465
CASH_IN     1949
CASH_OUT    1321
TRANSFER     921
DEBIT        344
Name: count, dtype: int64

In [12]:
df["step"].min(), df["step"].max()

(np.int64(1), np.int64(7))

In [13]:
df["step"].nunique()

7

## LOADING FULL DATASET FOR FURTHER ANALYSIS


In [14]:
import pandas as pd

In [15]:
file_path = "../dataset/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

In [16]:
df.shape

(6362620, 11)

In [17]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

## placing transaction in cronological order 


In [18]:
df = df.sort_values("step").reset_index(drop=True)

## feature engineering:- 

In [19]:
df["balance_change_orig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]

df["balance_change_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]

df["amount_to_orig_balance"] = (
    df["amount"] / (df["oldbalanceOrg"] + 1)
)

In [20]:
"""creating trasaction event id"""

df.insert(0, "event_id", range(len(df)))

In [21]:
df.head()

,event_id,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_change_orig,balance_change_dest,amount_to_orig_balance
0,0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.0,0,0,9839.64,0.0,0.057834
1,1,1,PAYMENT,5157.05,C1562950869,1667.92,0.00,M2021835850,0.0,0.0,0,0,1667.92,0.0,3.090052
2,2,1,PAYMENT,5746.44,C845388562,0.00,0.00,M550572371,0.0,0.0,0,0,0.00,0.0,5746.440000
3,3,1,PAYMENT,5607.36,C948424584,5202.00,0.00,M1447685190,0.0,0.0,0,0,5202.00,0.0,1.077717
4,4,1,PAYMENT,6360.79,C2027701910,3731.00,0.00,M1345293143,0.0,0.0,0,0,3731.00,0.0,1.704392


In [22]:
df = pd.get_dummies(
    df,
    columns=["type"],
    dtype=int
)

In [23]:
df = df.drop(columns=["isFlaggedFraud"])

In [24]:
df.columns.tolist()

['event_id',
 'step',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'balance_change_orig',
 'balance_change_dest',
 'amount_to_orig_balance',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER']

In [25]:
df.isnull().sum()

event_id                  0
step                      0
amount                    0
nameOrig                  0
oldbalanceOrg             0
newbalanceOrig            0
nameDest                  0
oldbalanceDest            0
newbalanceDest            0
isFraud                   0
balance_change_orig       0
balance_change_dest       0
amount_to_orig_balance    0
type_CASH_IN              0
type_CASH_OUT             0
type_DEBIT                0
type_PAYMENT              0
type_TRANSFER             0
dtype: int64

In [26]:
df.isnull().sum().sum()

np.int64(0)

In [27]:
import numpy as np

In [28]:
"""checking infinite values """

numeric_columns = df.select_dtypes(include=np.number).columns

np.isinf(df[numeric_columns]).sum().sum()

np.int64(0)

In [29]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,event_id,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,balance_change_orig,balance_change_dest,amount_to_orig_balance,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,0,1,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,9839.64,0.00,0.057834,0,0,0,1,0
1,1,1,5157.05,C1562950869,1667.92,0.00,M2021835850,0.00,0.00,0,1667.92,0.00,3.090052,0,0,0,1,0
2,2,1,5746.44,C845388562,0.00,0.00,M550572371,0.00,0.00,0,0.00,0.00,5746.440000,0,0,0,1,0
3,3,1,5607.36,C948424584,5202.00,0.00,M1447685190,0.00,0.00,0,5202.00,0.00,1.077717,0,0,0,1,0
4,4,1,6360.79,C2027701910,3731.00,0.00,M1345293143,0.00,0.00,0,3731.00,0.00,1.704392,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6362615,6362615,743,339682.13,C2013999242,339682.13,0.00,C1850423904,0.00,0.00,1,339682.13,0.00,0.999997,0,0,0,0,1
6362616,6362616,743,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,339682.13,339682.13,0.999997,0,1,0,0,0
6362617,6362617,743,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,6311409.28,0.00,1.000000,0,0,0,0,1
6362618,6362618,743,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,6311409.28,6311409.27,1.000000,0,1,0,0,0


In [30]:
df.isnull().sum().sum()

np.int64(0)

In [31]:
df[numeric_columns] = df[numeric_columns].fillna(0)

## fraud class distribution 

In [32]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [33]:
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [34]:
fraud_percentage = df["isFraud"].mean() * 100

print(f"Fraud percentage: {fraud_percentage:.4f}%")

Fraud percentage: 0.1291%


## checking duplicate recordes

In [35]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [36]:
"""checking cronological order """

print("Chronological order:", df["step"].is_monotonic_increasing)

Chronological order: True


In [37]:
df.dtypes

event_id                    int64
step                        int64
amount                    float64
nameOrig                      str
oldbalanceOrg             float64
newbalanceOrig            float64
nameDest                      str
oldbalanceDest            float64
newbalanceDest            float64
isFraud                     int64
balance_change_orig       float64
balance_change_dest       float64
amount_to_orig_balance    float64
type_CASH_IN                int64
type_CASH_OUT               int64
type_DEBIT                  int64
type_PAYMENT                int64
type_TRANSFER               int64
dtype: object

In [38]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 6362620
Columns: 18


## saving the processed dataset 


In [39]:
import os

processed_folder = "../dataset/processed"

os.makedirs(processed_folder, exist_ok=True)

processed_path = "../dataset/processed/paysim_processed.csv"

df.to_csv(
    processed_path,
    index=False
)

print("Processed dataset saved successfully.")
print("File:", processed_path)

Processed dataset saved successfully.
File: ../dataset/processed/paysim_processed.csv


In [40]:
print("Processed file exists:", os.path.exists(processed_path))
print("Rows in original dataframe:", len(df))
print("Columns in processed dataframe:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("Fraud percentage:", f"{df['isFraud'].mean() * 100:.4f}%")
print("Chronological order:", df["step"].is_monotonic_increasing)

Processed file exists: True
Rows in original dataframe: 6362620
Columns in processed dataframe: 18
Missing values: 0
Fraud percentage: 0.1291%
Chronological order: True
